# G3 Parity — Vector Field on Three-Sample Plane

Visualizes score `s(x,σ) = (D-x)/σ²` and denoiser `D(x,σ)` on a 2D plane spanned by:
- **x_a** (cyan ★) — a training sample (memorized attractor)
- **x_b** (green ★) — a valid non-training sample (Hamming=2 from x_a, group-0 bits flipped in pairs)
- **x_c** (red ★) — an invalid sample (Hamming=1 from x_a, one bit flipped → group-0 parity broken)

Plane axes:
- α-axis: direction x_a → x_b (valid manifold direction)
- β-axis: component of x_c − x_a perpendicular to α (invalid direction)

## Checkpoints (G3 rep2)
| Checkpoint | mem ratio | mid-σ train/test gap | Interpretation |
|---|---|---|---|
| ep 58780  | ~0%  | ~0      | post rule-learning, pre-memorization |
| ep 242446 | ~1%  | 0.013   | train/test gap just starting |
| ep 345511 | ~1%  | 0.067   | gap clearly open, pre-mem onset |
| ep 492388 | 8%   | 0.134   | just after memorization onset |
| ep 701704 | 21%  | 0.159   | well into memorization |


In [ ]:
import sys
sys.path.insert(0, '/n/home12/binxuwang/Github/DiffusionAttnConsistency')

import numpy as np
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype']  = 42
matplotlib.rcParams['axes.spines.top']   = False
matplotlib.rcParams['axes.spines.right'] = False
import matplotlib.pyplot as plt

from core.vector_field_lib import (
    load_model, load_training_data,
    eval_field_on_grid,
    project_to_basis,
    plot_vector_field_2d,
    plot_denoiser_target_2d,
)

SAVEROOT = '/n/holylfs06/LABS/kempner_fellow_binxuwang/Users/binxuwang/DL_Projects/DiffusionParityLearning'
FIGDIR   = '/n/home12/binxuwang/Github/DiffusionAttnConsistency/figures/vector_field'
import os; os.makedirs(FIGDIR, exist_ok=True)

EXP        = 'DiT_mini_parity_N4096_D36_G3_even_rep2'
GROUP_SIZE = 3
DEVICE     = 'cpu'   # change to 'cuda' if available

## 1. Build the three anchor samples and 2D plane

In [ ]:
x_train = load_training_data(EXP, saveroot=SAVEROOT)   # (4096, 36)

# x_a: training sample (memorized attractor)
x_a = x_train[0].numpy().copy()

# x_b: valid non-training sample — flip 2 bits in group 0 (parity preserved since (-1)²=+1)
x_b = x_a.copy(); x_b[0] *= -1; x_b[1] *= -1
assert np.prod(x_b[:3]) == 1.0, "group-0 parity violated"

# x_c: invalid sample — flip 1 bit in group 0 (product changes sign)
x_c = x_a.copy(); x_c[0] *= -1
assert np.prod(x_c[:3]) == -1.0, "group-0 should be invalid"

print(f"x_a: {x_a[:6]}")
print(f"x_b: {x_b[:6]}  Hamming(a,b)={(x_a!=x_b).sum()}  valid")
print(f"x_c: {x_c[:6]}  Hamming(a,c)={(x_a!=x_c).sum()}  invalid (group-0 parity={np.prod(x_c[:3]):.0f})")

In [ ]:
# ── Plane basis vectors ────────────────────────────────────────────────────────
ab = (x_b - x_a).astype(np.float32)
v_ab = ab / np.linalg.norm(ab)                          # α-axis: toward x_b

ac      = (x_c - x_a).astype(np.float32)
ac_perp = ac - ac.dot(v_ab) * v_ab                      # remove component along v_ab
v_ac    = ac_perp / np.linalg.norm(ac_perp)             # β-axis: toward x_c (perp)

# Coordinates of x_c in (α, β)
xc_alpha = float(ac.dot(v_ab))
xc_beta  = float(np.linalg.norm(ac_perp))
print(f"Plane coords:  x_a=(0,0)  x_b=(1,0)  x_c=({xc_alpha:.2f},{xc_beta:.2f})")

# ── Grid ──────────────────────────────────────────────────────────────────────
NGRID    = 45
alpha_ax = np.linspace(-0.5, 1.5, NGRID, dtype=np.float32)
beta_ax  = np.linspace(-0.5, 1.5, NGRID, dtype=np.float32)
A, B     = np.meshgrid(alpha_ax, beta_ax, indexing='ij')

grid_x = (x_a[None,None,:]
           + A[:,:,None] * ab[None,None,:]
           + B[:,:,None] * ac_perp[None,None,:]).astype(np.float32)
print(f"Grid: {grid_x.shape}")

## 2. Helper: plot one checkpoint × all σ values

In [ ]:
def plot_ckpt(model, ckpt_label, sigmas=(0.2, 0.5, 1.0, 2.0),
              save_tag=None, quiver_scale=30):
    """Two-row figure: score magnitude (top) + denoiser pull (bottom) for each σ."""
    s = 5  # quiver stride
    fig, axes = plt.subplots(2, len(sigmas), figsize=(4.5*len(sigmas), 9))
    fig.suptitle(f'G3 rep2  {ckpt_label}\n'
                 f'Plane: x_a(train) → x_b(valid novel) × x_c(invalid)',
                 fontsize=10, fontweight='bold')

    for col, sigma in enumerate(sigmas):
        res  = eval_field_on_grid(model, grid_x, sigma, device=DEVICE)
        u_s, v_s = project_to_basis(res['score'], v_ab, v_ac)
        Du,  Dv  = project_to_basis(res['D'],     v_ab, v_ac)

        # ── score magnitude ──────────────────────────────────────────────────
        ax  = axes[0][col]
        mag = res['mag_score']
        im  = ax.pcolormesh(alpha_ax, beta_ax, mag.T, cmap='hot',
                            vmin=0, vmax=np.percentile(mag, 97),
                            shading='auto', rasterized=True)
        plt.colorbar(im, ax=ax, shrink=0.75, label='||score||')
        ax.quiver(A[::s,::s], B[::s,::s], u_s[::s,::s], v_s[::s,::s],
                  color='white', alpha=0.75, scale=quiver_scale)
        ax.plot(0,        0,        'c*', ms=14, label='x_a (train)',      zorder=10)
        ax.plot(1,        0,        'g*', ms=14, label='x_b (valid novel)',zorder=10)
        ax.plot(xc_alpha, xc_beta,  'r*', ms=14, label='x_c (invalid)',    zorder=10)
        ax.set_xlabel('α (→ valid novel)', fontsize=9)
        ax.set_ylabel('β (→ invalid)',     fontsize=9)
        ax.set_title(f'Score magnitude  σ={sigma}', fontsize=9)
        if col == 0: ax.legend(fontsize=7, loc='upper right')

        # ── denoiser pull ────────────────────────────────────────────────────
        ax   = axes[1][col]
        clim = max(abs(Du.min()), abs(Du.max()))
        im2  = ax.pcolormesh(alpha_ax, beta_ax, Du.T, cmap='RdBu_r',
                             vmin=-clim, vmax=clim,
                             shading='auto', rasterized=True)
        plt.colorbar(im2, ax=ax, shrink=0.75, label='D · v_ab')
        ax.quiver(A[::s,::s], B[::s,::s],
                  (Du-A)[::s,::s], (Dv-B)[::s,::s],
                  color='k', alpha=0.6, scale=20)
        ax.plot(0, 0, 'c*', ms=14, zorder=10)
        ax.plot(1, 0, 'g*', ms=14, zorder=10)
        ax.plot(xc_alpha, xc_beta, 'r*', ms=14, zorder=10)
        ax.set_xlabel('α (→ valid novel)', fontsize=9)
        ax.set_ylabel('β (→ invalid)',     fontsize=9)
        ax.set_title(f'Denoiser pull D(x,σ)  σ={sigma}', fontsize=9)

    plt.tight_layout()
    if save_tag:
        for ext in ['pdf', 'png']:
            path = f'{FIGDIR}/G3_three_sample_plane_{save_tag}.{ext}'
            fig.savefig(path, bbox_inches='tight', dpi=150)
            print(f"Saved → {path}")
    return fig

## 3. ep 58780 — post rule-learning, pre-memorization

In [ ]:
m58k, _, _ = load_model(EXP, 58780, device=DEVICE, saveroot=SAVEROOT)
fig = plot_ckpt(m58k, 'ep 58780  (post rule-learning, pre-mem  |  mem~0%, gap~0)',
                save_tag='ep58780')
plt.show()

## 4. ep 242446 — train/test gap just starting

In [ ]:
m242k, _, _ = load_model(EXP, 242446, device=DEVICE, saveroot=SAVEROOT)
fig = plot_ckpt(m242k, 'ep 242446  (gap just starting  |  mem~1%, gap=0.013)',
                save_tag='ep242446')
plt.show()

## 5. ep 345511 — gap open, pre-memorization onset

In [ ]:
m345k, _, _ = load_model(EXP, 345511, device=DEVICE, saveroot=SAVEROOT)
fig = plot_ckpt(m345k, 'ep 345511  (gap open, pre-mem  |  mem~1%, gap=0.067)',
                save_tag='ep345511')
plt.show()

## 6. ep 492388 — just after memorization onset

In [ ]:
m492k, _, _ = load_model(EXP, 492388, device=DEVICE, saveroot=SAVEROOT)
fig = plot_ckpt(m492k, 'ep 492388  (just after mem onset  |  mem=8%, gap=0.134)',
                save_tag='ep492388')
plt.show()

## 7. ep 701704 — well into memorization

In [ ]:
m701k, _, _ = load_model(EXP, 701704, device=DEVICE, saveroot=SAVEROOT)
fig = plot_ckpt(m701k, 'ep 701704  (well into memorization  |  mem=21%, gap=0.159)',
                save_tag='ep701704')
plt.show()

## 8. Single-σ comparison across all checkpoints

In [ ]:
# Side-by-side score magnitude at σ=0.5 for all 5 checkpoints
ckpt_models = [
    ('ep 58k\npre-mem',    m58k),
    ('ep 242k\ngap start', m242k),
    ('ep 345k\ngap open',  m345k),
    ('ep 492k\nmem onset', m492k),
    ('ep 701k\nmem 21%',   m701k),
]
sigma_sel = 0.5
s = 5

fig, axes = plt.subplots(2, len(ckpt_models), figsize=(4*len(ckpt_models), 8))
fig.suptitle(f'G3 rep2  σ={sigma_sel}  — score & denoiser across checkpoints', fontsize=11, fontweight='bold')

for col, (lbl, model) in enumerate(ckpt_models):
    res = eval_field_on_grid(model, grid_x, sigma_sel, device=DEVICE)
    u_s, v_s = project_to_basis(res['score'], v_ab, v_ac)
    Du, Dv   = project_to_basis(res['D'],     v_ab, v_ac)

    ax = axes[0][col]
    mag = res['mag_score']
    im = ax.pcolormesh(alpha_ax, beta_ax, mag.T, cmap='hot',
                       vmin=0, vmax=np.percentile(mag, 97), shading='auto', rasterized=True)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.quiver(A[::s,::s], B[::s,::s], u_s[::s,::s], v_s[::s,::s],
              color='white', alpha=0.7, scale=30)
    ax.plot(0, 0, 'c*', ms=12, zorder=10)
    ax.plot(1, 0, 'g*', ms=12, zorder=10)
    ax.plot(xc_alpha, xc_beta, 'r*', ms=12, zorder=10)
    ax.set_title(lbl, fontsize=9)
    ax.set_xlabel('α', fontsize=8); ax.set_ylabel('β', fontsize=8)
    if col == 0: ax.set_ylabel('Score mag\nβ', fontsize=8)

    ax = axes[1][col]
    clim = max(abs(Du.min()), abs(Du.max()))
    im2 = ax.pcolormesh(alpha_ax, beta_ax, Du.T, cmap='RdBu_r',
                        vmin=-clim, vmax=clim, shading='auto', rasterized=True)
    plt.colorbar(im2, ax=ax, shrink=0.8)
    ax.quiver(A[::s,::s], B[::s,::s], (Du-A)[::s,::s], (Dv-B)[::s,::s],
              color='k', alpha=0.5, scale=20)
    ax.plot(0, 0, 'c*', ms=12, zorder=10)
    ax.plot(1, 0, 'g*', ms=12, zorder=10)
    ax.plot(xc_alpha, xc_beta, 'r*', ms=12, zorder=10)
    ax.set_xlabel('α', fontsize=8)
    if col == 0: ax.set_ylabel('Denoiser D·v_ab\nβ', fontsize=8)

plt.tight_layout()
for ext in ['pdf', 'png']:
    fig.savefig(f'{FIGDIR}/G3_checkpoint_comparison_sigma{sigma_sel}.{ext}',
                bbox_inches='tight', dpi=150)
plt.show()